<center><span style="font-size:40px;"><b>WORK FLOW IDEA</b></span></center>


We want to compare HMC vs NUTS when sampling from a known posterior, in this case a gaussian. So we need:
1. A model with a known posterior
2. Use *PyMC* to sample from it with both HMC and NUTS
3. Compare performance

#### STEP 1 — Generate synthetic data

We simulate data from a known distribution:
$$
x_i \sim \mathcal{N}(\mu_{\text{true}}, \sigma_{\text{true}})
$$

Conceptually we pretend the true parameters are unknown and we try to recover them via Bayesian inference.

#### STEP 2 — Define the model

Likelihood:

$$
p(x | \mu) = \prod_i \mathcal{N}(x_i | \mu, \sigma)
$$

Prior:

$$
\mu \sim \mathcal{N}(0, 10)
$$

Posterior:

$$
p(\mu | x) \propto p(x | \mu) p(\mu)
$$

This posterior is analytically tractable so we then can compare MCMC results with true posterior and check correctness.

#### STEP 3 — Define model in PyMC

Example:
```python
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=10)
    
    x_obs = pm.Normal("x_obs", mu=mu, sigma=sigma_true, observed=data)
```

#### STEP 4 — Run BOTH samplers

```python
# NUTS
trace_nuts = pm.sample(step=pm.NUTS(), draws=2000, tune=1000)

# HMC
trace_hmc = pm.sample(step=pm.HamiltonianMC(), draws=2000, tune=1000)
```

HMC requires tuning of the two parameters:
* step size $\epsilon $
* number of steps $L$

and we achieve it through varying:
```python
pm.HamiltonianMC(step_scale=0.1, path_length=1.0)
```

#### STEP 5 — Compare results

We compare posterior correctness (with analytical solutions):
* mean
* std

We compare convergence diagnostics (through ArviZ):

```python
az.summary(trace_nuts)
az.summary(trace_hmc)

# autocorrelation (the lower the better)
az.plot_autocorr(trace_nuts)
az.plot_autocorr(trace_hmc)

# trace plots
az.plot_trace(trace_nuts)
az.plot_trace(trace_hmc)
```

We look at:
* ESS (effective sample size)
* R-hat
* efficiency

$$
\text{Efficiency} = \frac{\text{ESS}}{\text{computation cost}}
$$

We can approximate:
* ESS / second
* ESS / number of samples

#### STEP 6 — Repeat with harder problem

Case 1: Unknown variance
$$
x_i \sim \mathcal{N}(\mu, \sigma), \quad \sigma \text{ unknown}
$$

Now:
* dimension = 2
* harder posterior

Case 2: Multivariate Gaussian (like in the paper)
$$
\theta \sim \mathcal{N}(0, \Sigma)
$

### CONCLUSIONS

We want to show that: 
* HMC:
    * sensitive to parameters ( \epsilon, L )
    * can perform poorly if badly tuned
* NUTS:
    * automatically adapts trajectory length
    * more robust

like what the paper claims.